# Sonata congelado + cabeza MLP de tres clases

Entrena y evalúa un detector de **postes de BT y de MT** sobre los bloques etiquetados de
`datos/crudo/etiquetado/`. El backbone es **Sonata (PTv3) congelado**: sus rasgos por voxel ya están en
`datos/intermedio/cache/sonata/` (los genera `codigo/postes/preparar_sonata.py`, celda 1). Aquí se entrena solo la
**cabeza MLP no lineal** con **tres salidas** (fondo, `poste_bt`, `poste_mt`) y se mide lo que
importa al inventario: **postes hallados, postes inventados y postes con la clase cambiada**.

Este cuaderno es uno de la familia (ver `etapa_4_5_modelado_evaluacion/README.md`): cada modelo tiene su
cuaderno, y todos miden igual porque la evaluación por objeto vive en `evaluacion_postes.py`.
La detección y la clase se separan a propósito: un voxel es *poste* si `1 − p(fondo)` supera
el umbral; los voxeles de poste se agrupan por columna y pasan el filtro de forma; y cada
**objeto** recibe la clase (BT o MT) con más probabilidad sumada en sus voxeles.

Lo que hace, en orden:

1. **Preparación** — comprueba `datos/intermedio/cache/sonata/`; lanza `codigo/postes/preparar_sonata.py` si falta un bloque o
   si un recorte es más nuevo que su cache (entonces solo se reetiqueta, sin GPU).
2. **Datos** — qué hay en cada bloque y cómo se reparte en *train / val / test* **por bloque**,
   según la carpeta en que esté cada nube (`datos/crudo/etiquetado/<escaneo>/train/` o `.../test/`).
3. **Entrenamiento** de la cabeza con curvas de pérdida, AP por clase y F1 por época y parada temprana.
4. **Umbral** elegido en validación con la métrica de detección por objeto.
5. **Test** en bloques que el modelo no vio: detección por bloque y tabla por clase.
6. **Salidas** — `.laz` con las predicciones (clase 5 o 12) para CloudCompare, CSV con enlaces a
   Street View, modelo y `resumen_<modelo>.json` en `etapa_4_5_modelado_evaluacion/resultados/postes/corrida<N>_<fecha>_<modelo>/`, una carpeta por corrida.
7. **Comparación** con los demás cuadernos que hayan dejado su `resumen_*.json`.

> Entorno: `F:\venvs\sonata` (kernel `py312-sonata`). Antes de abrir el cuaderno conviene tener
> `CUDA_PATH=F:\venvs\cuda_shim` en el entorno; la celda siguiente lo fija si falta.

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "etapa_4_5_modelado_evaluacion").is_dir())
AQUI = RAIZ / "etapa_4_5_modelado_evaluacion" / "codigo" / "postes"
os.chdir(AQUI); sys.path.insert(0, str(AQUI))
os.environ.setdefault("CUDA_PATH", r"F:\venvs\cuda_shim")   # spconv en Windows lo necesita

import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_recall_fscore_support

import postes_datos as pdt
from postes_datos import cargar_cache_sonata, contar_postes
from evaluacion_postes import (barrido_umbral, elegir_umbral, graficar_umbral, evaluar_conjunto,
                               escribir_salidas, comparar_resumenes)

SEMILLA = 0
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
# POSTES_PRUEBA_RAPIDA=1 reduce la particion y las epocas: es para comprobar que el
# cuaderno corre de punta a punta, no para sacar numeros. Escribe en resultados/postes_prueba_rapida/<modelo>/.
PRUEBA_RAPIDA = os.environ.get("POSTES_PRUEBA_RAPIDA") == "1"
CACHE_S = pdt.CACHE_SONATA
SALIDA_BASE = RAIZ / "etapa_4_5_modelado_evaluacion" / "resultados" / ("postes_prueba_rapida" if PRUEBA_RAPIDA else "postes")
# En Khipu (scripts/khipu/) cada job escribe en su propia carpeta: POSTES_SALIDA la fija.
if os.environ.get("POSTES_SALIDA"):
    SALIDA_BASE = Path(os.environ["POSTES_SALIDA"])

# Los scripts de preparacion corren SIEMPRE con el python del entorno sonata (spconv,
# sonata, laspy...), aunque el cuaderno se haya abierto con otro kernel por descuido.
VENV_SONATA = Path(r"F:\venvs\sonata\Scripts\python.exe")
PY = str(VENV_SONATA) if VENV_SONATA.exists() else sys.executable
if VENV_SONATA.exists() and Path(sys.executable).resolve() != VENV_SONATA.resolve():
    print(f"AVISO: este kernel es {sys.executable}\n       y no el entorno sonata. Cambia el kernel a "
          "'Python 3.12 (sonata)': el entrenamiento necesita su torch con CUDA.")

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"kernel {sys.executable}\npython {sys.version.split()[0]} · torch {torch.__version__} · {DEV}"
      + (f" · {torch.cuda.get_device_name(0)}" if DEV == "cuda" else "") + f"\nCUDA_PATH={os.environ['CUDA_PATH']}")
print("prueba rapida" if PRUEBA_RAPIDA else "modo completo")

## Parámetros

Todo lo que se puede querer tocar está aquí. **La partición la dicen las carpetas**: cada bloque
va a *train* o a *test* según estén sus recortes en `datos/crudo/etiquetado/<escaneo>/train/` o en `datos/crudo/etiquetado/<escaneo>/test/`
(mover un bloque de carpeta basta; las caches se identifican por nombre de bloque y no se
rehacen). Validación no tiene carpeta: se apartan de *train* los bloques de `VAL_DE_TRAIN`, que
el modelo no ve y sirven para la parada temprana y el umbral. Por defecto son `b_1494_43241` y
`b_1497_43238` (8 y 6 postes de BT), los mismos que en las corridas anteriores.

Los postes de MT etiquetados están en dos bloques: **siete en `b_1465_43337`** (train) y uno en
`b_1494_43240` (test). Un solo poste de MT en test es poco para medir esa clase: la cifra de MT se
lee como indicio, no como estimación.

In [ ]:
# Un modelo por clase: "BT" o "MT". La particion depende de la clase, por eso va primero.
CLASE_OBJETIVO  = os.environ.get("POSTES_CLASE", "MT")   # POSTES_CLASE la fija el job de Khipu
# 10-nov-2026: la particion es POR CLASE. El modelo BT la toma de
# datos/crudo/etiquetado/<escaneo>/poste_bt/<train|test>/ y el MT de .../poste_mt/<train|test>/;
# un bloque puede estar en train para una clase y en test para la otra.
VAL_DE_TRAIN = list(pdt.VAL_DE_TRAIN[CLASE_OBJETIVO])   # bloques de train que se apartan para validar
PARTICION = pdt.particion_por_carpetas(CLASE_OBJETIVO, VAL_DE_TRAIN)
if PRUEBA_RAPIDA:
    PARTICION = {"BT": {"train": ["AVP01MT2611202501_b_1465_43337", "PACHAP01MT0108202501_b_1494_43242"],
                        "val": ["PACHAP01MT0108202501_b_1497_43238"], "test": ["PACHAP01MT0108202501_b_1494_43240"]},
                 "MT": {"train": ["AVP01MT2611202501_b_1465_43337"],
                        "val": ["PACHAP01MT0108202502_b_1489_43238"], "test": ["PACHAP01MT0108202502_b_1489_43239"]},
                 }[CLASE_OBJETIVO or "BT"]
for k, v in PARTICION.items():
    assert v, (f"la particion '{k}' esta vacia: revisa datos/crudo/etiquetado/<escaneo>/poste_{(CLASE_OBJETIVO or 'bt').lower()}/"
               "train y test, y VAL_DE_TRAIN")

MAX_FONDO_TRAIN = 400_000     # voxeles de fondo POR BLOQUE en el entrenamiento (los de poste entran todos)
MAX_FONDO_VAL   = 200_000     # idem para la muestra de validacion de las curvas
EPOCAS          = 8 if PRUEBA_RAPIDA else 60
PACIENCIA       = 20          # epocas sin mejorar el AP de validacion antes de parar (10 hasta la corrida 6)
LOTE            = 8192
LR, WD          = 5e-5, 1e-3  # 7-nov-2026: lr 5e-4 -> 5e-5. Con 5e-4 la mejor epoca fue SIEMPRE
                              # la 1 (corridas 5-8): la cabeza aprendia lo util en los ~500 pasos
                              # de la primera epoca y despues el AP de validacion solo bajaba, asi
                              # que la parada temprana no tenia donde elegir. Con 10x menos lr el
                              # punto bueno deberia caer varias epocas despues.
OCULTAS         = (256, 128)  # Corrida 1 (30-oct). La version regularizada (128, wd 1e-2,
DROPOUT         = 0.3         # dropout 0.5; corrida 2) hallo MENOS postes en test (33-38 de 48
                              # frente a 40) con la misma validacion: el ranking es lo que
                              # cuenta y esta cabeza lo hace mejor.
# Negativos dificiles: tras entrenar, se puntuan los bloques de train ENTEROS y los
# voxeles de fondo con mas puntuacion de poste (troncos, senales, aristas: lo que se
# confunde con un poste) se anaden a la muestra y se reentrena desde cero. 0 lo desactiva.
RONDAS_NEGATIVOS_DUROS = 1
NEG_DUROS_POR_BLOQUE   = 50_000
# Franja neutra (6-nov-2026): los voxeles de fondo a menos de FRANJA_NEUTRA m en planta de un
# poste etiquetado no entran al entrenamiento (ni como fondo ni como negativo duro). Los recortes
# dejan fuera puntas o caras del poste; sin esto el modelo aprende que eso es fondo y los
# negativos duros lo refuerzan. No cambia las etiquetas ni la metrica por objeto. 0 lo desactiva.
FRANJA_NEUTRA          = 0.3
# Postes sin etiquetar (14-nov-2026): al minar negativos duros, el fondo que el modelo puntua
# alto (>= UMBRAL_SIN_ETIQUETA) y forma un objeto con forma de poste (filtro de forma) es casi
# siempre un poste que no se etiqueto. Se saca de los negativos duros, con RADIO_SIN_ETIQUETA m
# en planta alrededor; sin esto la ronda 2 le ensena con fuerza que un poste es fondo. Se listan
# en candidatos_sin_etiqueta.csv para revisarlos y etiquetarlos. 0 desactiva el umbral.
UMBRAL_SIN_ETIQUETA    = 0.5
RADIO_SIN_ETIQUETA     = 0.5
# 7-nov-2026: se anaden 0,97-0,99. La corrida 7 y el humo eligieron 0,95, el tope de la rejilla.
UMBRALES        = np.round(np.concatenate([[0.01, 0.02, 0.03], np.arange(0.05, 0.96, 0.05), [0.97, 0.98, 0.99]]), 2)
# Un modelo por clase (6-nov-2026): primero solo BT, luego otro entrenamiento solo MT.
# Los postes de la otra clase no entran al entrenamiento (ni como poste ni como fondo) y en la
# evaluacion no se buscan ni penalizan. Cambia a "MT" para el segundo entrenamiento.
# (CLASE_OBJETIVO se fija al principio de esta celda.)
# Curva de aprendizaje (10-nov-2026): fraccion de los postes de train que se usa. Los demas
# quedan fuera, como la franja neutra: ni poste ni fondo. Validacion y test no cambian.
# 1 = todos. etapa_4_5_modelado_evaluacion/scripts/khipu/curva.slurm las recorre con varias semillas.
FRACCION_POSTES  = float(os.environ.get("POSTES_FRACCION", "1"))
SEMILLA_FRACCION = int(os.environ.get("POSTES_SEMILLA_FRACCION", "0"))
CLASES          = (pdt.CLASE_POR_SUFIJO[CLASE_OBJETIVO],)
# Rasgos de altura (altura sobre el suelo, altura de la columna, posicion en ella) junto a
# los de Sonata: entradas para que la red aprenda de las etiquetas que alto es cada poste.
RASGOS_ALTURA   = True
ETIQUETA_MODELO = f"sonata_mlp_solo_{CLASE_OBJETIVO.lower()}"
# cada corrida en su carpeta: resultados/postes/corrida<N>_<fecha>_<modelo>/
SALIDA = pdt.nueva_corrida(ETIQUETA_MODELO, SALIDA_BASE, fija=PRUEBA_RAPIDA)
print("salidas ->", SALIDA.relative_to(RAIZ))

# Las clases que detecta la cabeza, ademas del fondo (columna 0 de las probabilidades).
NOMBRES = ["fondo"] + [pdt.NOMBRE_CLASE[c] for c in CLASES]
INDICE  = np.zeros(256, dtype="i8")                 # codigo de clase -> columna de la cabeza
for i, c in enumerate(CLASES):
    INDICE[c] = i + 1

print(json.dumps(PARTICION, indent=1)); print("clases:", NOMBRES)

## 1 · Preparación de la cache de Sonata

`codigo/postes/preparar_sonata.py` pasa Sonata por cada bloque y guarda los rasgos (1088 → 256 con PCA) en
`datos/intermedio/cache/sonata/`. Salta los bloques que ya están y, si un **recorte es más nuevo que la cache**
de su bloque, recalcula **solo la etiqueta**, sin GPU. La primera vez son del orden de
**2-4 min por bloque** (GPU); reetiquetar tarda segundos.

In [ ]:
inv = pdt.inventario()

def estado(bloque, marca):
    if not marca.exists():
        return "falta"
    fila = inv[inv["bloque"] == bloque].iloc[0].to_dict()
    return "etiquetas viejas" if pdt.etiquetas_desactualizadas(fila, marca) else "ok"

inv["sonata"] = inv["bloque"].map(lambda b: estado(b, CACHE_S / b / "meta.json"))
inv["particion"] = inv["bloque"].map({b: k for k, bs in PARTICION.items() for b in bs})
usados = inv[inv["particion"].notna()]
fuera = inv["particion"].isna() & inv[CLASE_OBJETIVO].notna()
if fuera.any():
    print(f"con recorte {CLASE_OBJETIVO} pero fuera de la particion:", inv.loc[fuera, "bloque"].tolist())
display(usados[["bloque", "escaneo", f"conjunto_{CLASE_OBJETIVO}", "particion", "BT", "MT", "sonata"]]
        .assign(BT=lambda d: d["BT"].notna(), MT=lambda d: d["MT"].notna()))

def lanzar(script, bloques):
    if not bloques:
        return
    print(f"-> {PY} {script} {' '.join(bloques)}")
    env = {**os.environ, "CUDA_PATH": os.environ.get("CUDA_PATH", r"F:\venvs\cuda_shim"), "PYTHONIOENCODING": "utf-8"}
    p = subprocess.Popen([PY, "-u", str(AQUI / script), *bloques], cwd=str(AQUI), env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    cola = []
    for linea in p.stdout:
        if "RuntimeWarning" in linea or "relleno =" in linea:
            continue
        print("   " + linea.rstrip()); cola = (cola + [linea.rstrip()])[-25:]
    if p.wait() != 0:
        raise RuntimeError(f"{script} termino con codigo {p.returncode}. Ultimas lineas:\n" + "\n".join(cola))

# El script decide por bloque: si falta, lo prepara entero; si solo tiene etiquetas viejas, reetiqueta.
lanzar("preparar_sonata.py", usados.loc[usados["sonata"] != "ok", "bloque"].tolist())

## 2 · Los datos

Una fila por bloque: voxeles útiles (HAG entre 0,3 y 40 m), voxeles de poste y **cuántos
postes** de cada clase son, contando objetos (componentes conexas a 0,6 m de los voxeles
etiquetados). Es el número que da sentido a las métricas de después: con 8 postes de MT en total,
cada uno es un 12 % de recuperación de esa clase.

In [ ]:
filas = []
for parte, bloques in PARTICION.items():
    for b in bloques:
        d = cargar_cache_sonata(b)
        n_post = contar_postes(d["P"], d["Y"], (5, 12))   # las dos clases, aunque el modelo use una
        filas.append({"particion": parte, "bloque": b, "escaneo": d["escaneo"][:6],
                      "voxeles": len(d["Y"]), "vox_poste_bt": int((d["Y"] == 5).sum()), "vox_poste_mt": int((d["Y"] == 12).sum()),
                      "postes_bt": n_post.get(5, 0), "postes_mt": n_post.get(12, 0),
                      "dim": d["F"].shape[1], "GB": round(d["F"].nbytes / 1e9, 2)})
tabla_datos = pd.DataFrame(filas)
display(tabla_datos)
display(tabla_datos.groupby("particion")[["voxeles", "vox_poste_bt", "vox_poste_mt", "postes_bt", "postes_mt"]].sum())
ETIQUETAS = pdt.huella_etiquetas(tabla_datos["bloque"])
print(f"etiquetas: {ETIQUETAS['n_recortes']} recortes, huella {ETIQUETAS['md5'][:12]}")

## 3 · Muestras de entrenamiento y validación

Entran **todos** los voxeles de `poste_bt` y `poste_mt` y, por bloque, hasta `MAX_FONDO_TRAIN`
voxeles de fondo al azar: el fondo es el 98 % de los voxeles y no hace falta verlo entero para
aprenderlo. La etiqueta tiene **tres valores** (0 fondo, 1 BT, 2 MT). Los rasgos se estandarizan
con la media y desvío del conjunto de entrenamiento, que se guardan con el modelo.

In [ ]:
def cargar(b):
    "Bloque de la cache de Sonata con sus rasgos de altura (A) al lado."
    d = cargar_cache_sonata(b)
    d["A"] = (pdt.rasgos_altura(d["P"], d["hag"]) if RASGOS_ALTURA
              else np.zeros((len(d["Y"]), 0), dtype="f4"))
    return d

def rasgos(d, idx):
    "Rasgos de Sonata (256) mas los de altura, de los voxeles idx (indices o slice)."
    return np.hstack([np.asarray(d["F"][idx], dtype="f4"), d["A"][idx]])

def postes_fuera(bloques):
    "Mascara por bloque de los voxeles de los postes de train que se dejan fuera (FRACCION_POSTES < 1)."
    if FRACCION_POSTES >= 1:
        return {}, None
    objetos, largos = [], {}
    for b in bloques:
        d = cargar_cache_sonata(b)
        largos[b] = len(d["Y"])
        objetos += [(b, o) for o in pdt.objetos_columna(d["P"], np.isin(d["Y"], CLASES))]
    n_usar = max(1, round(FRACCION_POSTES * len(objetos)))
    usar = set(np.random.default_rng(SEMILLA_FRACCION).choice(len(objetos), n_usar, replace=False).tolist())
    fuera = {b: np.zeros(n, dtype=bool) for b, n in largos.items()}
    for k, (b, o) in enumerate(objetos):
        if k not in usar:
            fuera[b][o] = True
    print(f"curva de aprendizaje: {n_usar} de {len(objetos)} postes de train (fraccion {FRACCION_POSTES}, semilla {SEMILLA_FRACCION})")
    return fuera, (n_usar, len(objetos))

FUERA, POSTES_USADOS = postes_fuera(PARTICION["train"])

def muestra(bloques, max_fondo, semilla=SEMILLA):
    rng = np.random.default_rng(semilla)
    F, Y = [], []
    for b in bloques:
        d = cargar(b)
        poste = np.isin(d["Y"], CLASES)
        if b in FUERA:
            poste &= ~FUERA[b]
        idx = np.flatnonzero(poste)
        fondo = np.flatnonzero((d["Y"] == 0) & ~pdt.franja_neutra(d["P"], d["Y"], FRANJA_NEUTRA))
        if len(fondo) > max_fondo:
            fondo = rng.choice(fondo, max_fondo, replace=False)
        idx = np.sort(np.concatenate([idx, fondo]))
        F.append(rasgos(d, idx)); Y.append(d["Y"][idx])
    return np.concatenate(F), np.concatenate(Y)

t0 = time.perf_counter()
F_tr, Y_tr = muestra(PARTICION["train"], MAX_FONDO_TRAIN)
F_va, Y_va = muestra(PARTICION["val"], MAX_FONDO_VAL, semilla=SEMILLA + 1)
y_tr, y_va = INDICE[Y_tr], INDICE[Y_va]          # 0 fondo, 1 la clase objetivo

MEDIA = F_tr.astype("f4").mean(0); DESVIO = F_tr.astype("f4").std(0) + 1e-6
def estandarizar(F):
    # np.array copia: los trozos de memmap son de solo lectura y torch se queja
    return ((torch.from_numpy(np.array(F, dtype="f4")) - torch.from_numpy(MEDIA)) / torch.from_numpy(DESVIO))

Xtr, Xva = estandarizar(F_tr).to(DEV).half(), estandarizar(F_va).to(DEV).half()
ytr, yva = torch.from_numpy(y_tr).to(DEV), torch.from_numpy(y_va).to(DEV)
def cuenta(y):
    return " · ".join(f"{NOMBRES[i]} {int((y == i).sum()):,}" for i in range(len(NOMBRES)))
print(f"train {len(ytr):,} voxeles ({cuenta(y_tr)})\nval   {len(yva):,} voxeles ({cuenta(y_va)})\n"
      f"{F_tr.shape[1]} rasgos · {time.perf_counter() - t0:.0f} s")
del F_tr, F_va

## 4 · La cabeza MLP y su entrenamiento

Dos capas ocultas con GELU y *dropout* sobre los 256 rasgos de Sonata y **tres salidas**.
Pérdida: entropía cruzada con peso por clase `√(fondo/clase)`, el mismo compromiso que en el
caso binario (la inversa pura hundía la precisión; MT, con muy pocos voxeles, recibe el peso más
alto). Por época se guarda la pérdida de entrenamiento, la de validación, el **AP por clase**
(área bajo precisión-recuperación, no depende del umbral), el AP de *poste contra fondo* y el F1 de
poste con la clase más probable. Se para cuando el AP medio de validación (sobre las clases que
hay en validación) lleva `PACIENCIA` épocas sin mejorar y se recupera la mejor época.

**Negativos difíciles.** El fondo al azar es casi todo fachada, suelo alto y vegetación fácil;
lo que el modelo confunde con un poste (troncos, señales, aristas de edificio) es raro en la
muestra. Tras la primera ronda se puntúan los bloques de entrenamiento enteros, se toman los
`NEG_DUROS_POR_BLOQUE` voxeles de fondo con más puntuación de poste y se reentrena desde cero con
ellos añadidos. Gana la ronda con mejor AP de validación.

> Si validación no tiene postes de MT (la partición por defecto no los tiene), el AP de MT sale
> vacío y no entra en la parada temprana: la clase MT solo se mide en test, con un poste.

In [ ]:
class CabezaMLP(nn.Module):
    def __init__(self, d_in, n_clases=len(NOMBRES), ocultas=OCULTAS, dropout=DROPOUT):
        super().__init__()
        capas, d = [], d_in
        for h in ocultas:
            capas += [nn.Linear(d, h), nn.GELU(), nn.Dropout(dropout)]
            d = h
        capas.append(nn.Linear(d, n_clases))
        self.red = nn.Sequential(*capas)
    def forward(self, x):
        return self.red(x)

@torch.no_grad()
def logits(modelo, X, lote=65536):
    modelo.eval()
    return torch.cat([modelo(X[a:a + lote].float()) for a in range(0, len(X), lote)])

@torch.no_grad()
def puntuar_bloque(modelo, d, lote=400_000):
    "Probabilidades (n, 3) de TODOS los voxeles de un bloque, leyendo el memmap por trozos."
    modelo.eval()
    s = np.empty((len(d["Y"]), len(NOMBRES)), dtype="f4")
    for a in range(0, len(s), lote):
        s[a:a + lote] = torch.softmax(modelo(estandarizar(rasgos(d, slice(a, a + lote))).to(DEV)), dim=1).cpu().numpy()
    return s

def aps(y, p):
    "AP por clase (NaN si la clase no esta) y AP de poste contra fondo."
    out = {f"ap_{NOMBRES[i]}": (average_precision_score(y == i, p[:, i]) if (y == i).any() else np.nan)
           for i in range(1, len(NOMBRES))}
    out["ap_poste"] = average_precision_score(y > 0, 1 - p[:, 0])
    # Criterio de seleccion: AP de poste contra fondo. El promedio con MT (corrida 6) lo
    # dominaba el ruido de un AP de MT de ~0,1 y elegia siempre la epoca 1.
    out["ap_val"] = out["ap_poste"]
    return out

def entrenar(Xtr, ytr, Xva, yva, epocas=EPOCAS, nombre="ronda 1"):
    torch.manual_seed(SEMILLA)
    modelo = CabezaMLP(Xtr.shape[1]).to(DEV)
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epocas)
    n = torch.bincount(ytr, minlength=len(NOMBRES)).float()
    pesos = torch.where(n > 0, torch.sqrt(n[0] / n.clamp(min=1)), torch.ones_like(n)); pesos[0] = 1.0
    perdida = nn.CrossEntropyLoss(weight=pesos)
    historia, mejor, sin_mejora, t0 = [], {"ap": -1}, 0, time.perf_counter()
    print(f"{nombre}: {len(ytr):,} voxeles ({cuenta(ytr.cpu().numpy())}), pesos de clase "
          + ", ".join(f"{NOMBRES[i]} {float(pesos[i]):.1f}" for i in range(len(NOMBRES))))
    for ep in range(1, epocas + 1):
        modelo.train()
        perm = torch.randperm(len(Xtr), device=DEV)
        suma, m = 0.0, 0
        for a in range(0, len(perm), LOTE):
            b = perm[a:a + LOTE]
            loss = perdida(modelo(Xtr[b].float()), ytr[b])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            suma += float(loss) * len(b); m += len(b)
        sched.step()
        lg = logits(modelo, Xva)
        loss_va = float(perdida(lg, yva))
        p_va = torch.softmax(lg, dim=1).cpu().numpy()
        a_ = aps(y_va, p_va)
        pr, rc, f1, _ = precision_recall_fscore_support(y_va > 0, p_va.argmax(1) > 0, average="binary", zero_division=0)
        historia.append({"ronda": nombre, "epoca": ep, "loss_train": suma / m, "loss_val": loss_va, **a_,
                         "f1_val": f1, "precision_val": pr, "recall_val": rc, "lr": sched.get_last_lr()[0]})
        if a_["ap_val"] > mejor["ap"]:
            mejor = {"ap": a_["ap_val"], "epoca": ep, "p_va": p_va,
                     "estado": {k: v.detach().clone() for k, v in modelo.state_dict().items()}}
            sin_mejora = 0
        else:
            sin_mejora += 1
        print(f"  ep {ep:3d} · loss {suma / m:.4f} / val {loss_va:.4f} · AP val {a_['ap_val']:.3f} "
              "(" + " ".join(f"{k[3:]} {v:.3f}" for k, v in a_.items() if k != "ap_val") + f") · F1 poste {f1:.3f} "
              f"(P {pr:.2f} R {rc:.2f}) · {time.perf_counter() - t0:.0f} s" + ("  *" if sin_mejora == 0 else ""))
        if sin_mejora >= PACIENCIA:
            print(f"  parada temprana: {PACIENCIA} epocas sin mejorar el AP de validacion")
            break
    modelo.load_state_dict(mejor["estado"])
    mejor["pesos"] = pesos.cpu().numpy().tolist()
    return modelo, pd.DataFrame(historia), mejor

def sin_etiqueta(d, s):
    """Mascara del fondo que parece un poste sin etiquetar (y su entorno en planta), y una fila
    por objeto para revisarlo. Mismo filtro de forma que la evaluacion."""
    if UMBRAL_SIN_ETIQUETA <= 0:
        return np.zeros(len(s), dtype=bool), []
    alto = (d["Y"] == 0) & (s >= UMBRAL_SIN_ETIQUETA)
    forma = pdt.filtrar_columnas(d["P"], d["hag"], alto) if alto.any() else alto
    if not forma.any():
        return np.zeros(len(s), dtype=bool), []
    filas = []
    for g in pdt.objetos_columna(d["P"], forma):
        q = d["P"][g].astype("f8") + d["origen"]
        filas.append({"bloque": d["bloque"], "x": round(float(np.median(q[:, 0])), 2),
                      "y": round(float(np.median(q[:, 1])), 2), "z_cima": round(float(q[:, 2].max()), 2),
                      "alto_m": round(float(d["hag"][g].max() - d["hag"][g].min()), 2),
                      "cima_sobre_suelo_m": round(float(d["hag"][g].max()), 2),
                      "puntuacion_max": round(float(s[g].max()), 3), "voxeles": int(len(g))})
    marca = forma.astype(np.int8)
    zona = forma | pdt.franja_neutra(d["P"], marca, RADIO_SIN_ETIQUETA)
    return zona & (d["Y"] == 0), filas


CANDIDATOS_SIN_ETIQUETA = []

def negativos_duros(modelo, bloques, por_bloque):
    """Los voxeles de FONDO con mas puntuacion de poste de cada bloque de entrenamiento,
    sin los que parecen un poste sin etiquetar (ver sin_etiqueta)."""
    trozos = []
    for b in bloques:
        d = cargar(b)
        s = 1 - puntuar_bloque(modelo, d)[:, 0]
        excluir, filas = sin_etiqueta(d, s)
        CANDIDATOS_SIN_ETIQUETA.extend(filas)
        fondo = np.flatnonzero((d["Y"] == 0) & ~pdt.franja_neutra(d["P"], d["Y"], FRANJA_NEUTRA) & ~excluir)
        top = fondo[np.argsort(-s[fondo])[:por_bloque]]
        trozos.append(rasgos(d, np.sort(top)))
        print(f"  {b}: {len(top):,} negativos duros, puntuacion media {s[top].mean():.3f} (fondo global {s[fondo].mean():.4f})"
              + (f" · {len(filas)} posibles postes sin etiquetar fuera ({int(excluir.sum()):,} voxeles)" if filas else ""))
    if CANDIDATOS_SIN_ETIQUETA:
        t = pd.DataFrame(CANDIDATOS_SIN_ETIQUETA).sort_values(["bloque", "puntuacion_max"], ascending=[True, False])
        t.to_csv(SALIDA / "candidatos_sin_etiqueta.csv", index=False)
        print(f"  {len(t)} posibles postes sin etiquetar en train -> {SALIDA / 'candidatos_sin_etiqueta.csv'}")
    return np.concatenate(trozos)

rondas_hechas = []          # (modelo, historia, mejor) de cada ronda; gana la de mejor AP en validacion
modelo, historia, mejor = entrenar(Xtr, ytr, Xva, yva, nombre="ronda 1")
rondas_hechas.append((modelo, historia, mejor))
for r in range(RONDAS_NEGATIVOS_DUROS):
    print(f"\nminando negativos duros con el modelo de la ronda {r + 1}")
    Fnd = negativos_duros(modelo, PARTICION["train"], NEG_DUROS_POR_BLOQUE)
    Xtr = torch.cat([Xtr, estandarizar(Fnd).to(DEV).half()])
    ytr = torch.cat([ytr, torch.zeros(len(Fnd), device=DEV, dtype=ytr.dtype)])
    del Fnd
    modelo, historia, mejor = entrenar(Xtr, ytr, Xva, yva, nombre=f"ronda {r + 2}")
    rondas_hechas.append((modelo, historia, mejor))

historia_total = pd.concat([h for _, h, _ in rondas_hechas], ignore_index=True)
modelo, historia, mejor = max(rondas_hechas, key=lambda t: t[2]["ap"])
p_va, PESOS_CLASE = mejor["p_va"], mejor["pesos"]
display(historia_total.groupby("ronda", sort=False).agg(epocas=("epoca", "max"), mejor_ap_val=("ap_val", "max"),
                                                        loss_val_min=("loss_val", "min")).round(4))
RONDA_FINAL = historia["ronda"].iloc[0]
print(f"modelo final: {RONDA_FINAL}, mejor epoca {mejor['epoca']} con AP val {mejor['ap']:.3f}")

### Curvas de entrenamiento y validación

Lo que hay que mirar: si la pérdida de validación sube mientras la de entrenamiento baja, la
cabeza memoriza (más *dropout*, menos épocas o más bloques); si las dos bajan juntas y el AP
sigue subiendo al parar, faltan épocas. La línea vertical es la época que se conserva.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
rondas = list(historia_total.groupby("ronda", sort=False))
for i, (nombre, h) in enumerate(rondas):
    alpha = 1.0 if nombre == RONDA_FINAL else 0.35      # la ronda ganadora, en color pleno
    suf = "" if len(rondas) == 1 else f" · {nombre}"
    ax[0].plot(h["epoca"], h["loss_train"], color="C0", alpha=alpha, label=f"train{suf}")
    ax[0].plot(h["epoca"], h["loss_val"], color="C1", alpha=alpha, label=f"val{suf}")
    for col_, color_ in (("ap_poste_bt", "C2"), ("ap_poste_mt", "C6")):
        if col_ in h and h[col_].notna().any():
            ax[1].plot(h["epoca"], h[col_], color=color_, alpha=alpha, label=f"AP {col_[3:]}{suf}")
    ax[1].plot(h["epoca"], h["ap_poste"], "--", color="C7", alpha=alpha, label=f"AP poste vs fondo{suf}")
    ax[2].plot(h["epoca"], h["f1_val"], color="C3", alpha=alpha, label=f"F1 poste{suf}")
    if nombre == RONDA_FINAL:
        ax[2].plot(h["epoca"], h["precision_val"], "--", color="C4", alpha=.7, label="precisión poste")
        ax[2].plot(h["epoca"], h["recall_val"], "--", color="C5", alpha=.7, label="recuperación poste")
ax[0].set_title("Pérdida (entropía cruzada ponderada)"); ax[0].legend(fontsize=8)
ax[1].set_title("AP en validación"); ax[1].legend(fontsize=8)
ax[2].set_title("Validación por voxel, clase más probable"); ax[2].legend(fontsize=8)
for a in ax:
    a.axvline(mejor["epoca"], color="k", ls=":", lw=1); a.set_xlabel("época")
plt.tight_layout(); plt.savefig(SALIDA / f"curvas_{ETIQUETA_MODELO}.png"); plt.show()

plt.figure(figsize=(4.5, 4))
for i in range(1, len(NOMBRES)):
    if (y_va == i).any():
        prec, rec, _ = precision_recall_curve(y_va == i, p_va[:, i])
        plt.plot(rec, prec, label=f"{NOMBRES[i]} · AP {average_precision_score(y_va == i, p_va[:, i]):.3f}")
prec, rec, _ = precision_recall_curve(y_va > 0, 1 - p_va[:, 0])
plt.plot(rec, prec, "--", color="C7", label=f"poste vs fondo · AP {average_precision_score(y_va > 0, 1 - p_va[:, 0]):.3f}")
plt.xlabel("recuperación"); plt.ylabel("precisión"); plt.title("PR por voxel · validación"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5 · Umbral por métrica de objeto, en validación

El AP por voxel no es lo que paga el cliente: lo que cuenta es cuántos **postes** aparecen y
cuántos se inventan. Se puntúan los bloques de validación **enteros** (no la muestra); un voxel es
poste si `1 − p(fondo)` supera el umbral; los voxeles de poste se agrupan **por columna** y cada
objeto pasa el filtro de forma (`postes_datos.filtrar_columnas`). Luego se barre el umbral sobre
la **detección** (poste o no, sin mirar la clase) y se elige el que más postes halla con
precisión ≥ 0,5 (`evaluacion_postes.elegir_umbral`). La clase por objeto se evalúa después, en
test, con el umbral ya fijo.

In [ ]:
t0 = time.perf_counter()
datos_val = {b: cargar(b) for b in PARTICION["val"]}
COMPARTIDOS_VAL = pdt.marcar_compartidos(datos_val, PARTICION["train"], cargar_cache_sonata)
print(f"postes de validacion compartidos con train, fuera de la metrica: {COMPARTIDOS_VAL}")
probs_val = {b: puntuar_bloque(modelo, d) for b, d in datos_val.items()}
tabla_val = barrido_umbral(datos_val, probs_val, UMBRALES, CLASES)
UMBRAL = elegir_umbral(tabla_val)
print(f"umbral elegido: {UMBRAL} · {time.perf_counter() - t0:.0f} s")
display(tabla_val[tabla_val["filtro_forma"]].drop(columns="filtro_forma").round(3))
graficar_umbral(tabla_val, UMBRAL, SALIDA / f"umbral_{ETIQUETA_MODELO}.png")

## 6 · Test: bloques que el modelo no vio

Con el umbral fijado en validación, sin tocar nada más. Dos tablas:

- **Detección**, por bloque y en total: postes reales, hallados, candidatos y cuántos
  candidatos coinciden con un poste real (a 1 m), sin mirar la clase. También precisión y
  recuperación por voxel para quien quiera compararlas con la literatura.
- **Por clase**: de los postes reales de BT y de MT, cuántos los halló un candidato de **su
  clase** y cuántos solo uno de la otra (`hallados_otra_clase`); y de los candidatos de cada
  clase, cuántos son correctos, cuántos tocan un poste real de la **otra** clase
  (`clase_equivocada`) y cuántos no tocan ninguno (`inventados`).

In [ ]:
t0 = time.perf_counter()
datos_test = {b: cargar(b) for b in PARTICION["test"]}
COMPARTIDOS_TEST = pdt.marcar_compartidos(datos_test, PARTICION["train"] + PARTICION["val"], cargar_cache_sonata)
print(f"postes de test compartidos con train o validacion, fuera de la metrica: {COMPARTIDOS_TEST}")
probs_test = {b: puntuar_bloque(modelo, d) for b, d in datos_test.items()}
tabla_test, tabla_test_clases, clase_test = evaluar_conjunto(datos_test, probs_test, UMBRAL, CLASES)
print(f"TEST · umbral {UMBRAL} · {time.perf_counter() - t0:.0f} s\ndeteccion (poste o no):")
display(tabla_test)
print("por clase:")
display(tabla_test_clases)

# Diagnostico: lo mismo sobre los bloques de ENTRENAMIENTO. No es una metrica del modelo
# (los ha visto); sirve para distinguir "no aprende" (tambien falla aqui) de "no generaliza".
datos_train = {b: cargar(b) for b in PARTICION["train"]}
tabla_train, tabla_train_clases, _ = evaluar_conjunto(datos_train, {b: puntuar_bloque(modelo, d) for b, d in datos_train.items()},
                                                      UMBRAL, CLASES)
del datos_train
print("TRAIN (diagnostico, no es metrica):")
display(tabla_train.tail(1)); display(tabla_train_clases)

## 7 · Salidas

- `<bloque>_PRED_sonata_mlp_btmt.laz` — los voxeles predichos como poste en los bloques de
  test, con `classification` **5 (BT) o 12 (MT)** por objeto y los campos `confianza` (puntuación
  de poste), `p_bt`, `p_mt` y `hag`. Se abren en CloudCompare sobre el bloque original.
- `<bloque>_candidatos_sonata_mlp_btmt.csv` — un candidato por fila con su altura, clase
  predicha, la etiqueta del poste real más cercano (`fondo` si no hay: falso positivo o poste sin
  recortar) y un enlace a Street View. Los que **no** coinciden son los que hay que mirar.
- `mlp_sonata_mlp_btmt.pt` — la cabeza, la media/desvío de estandarización, las clases y el umbral.
- `resumen_sonata_mlp_btmt.json` — configuración, partición, huella MD5 de los recortes,
  historia por época, tablas de validación y de test (detección y por clase).

In [ ]:
cand = escribir_salidas(SALIDA, ETIQUETA_MODELO, datos_test, clase_test, probs_test, NOMBRES)
if len(cand):
    print("\ntest · clase predicha frente a la etiqueta del poste real mas cercano:")
    display(pd.crosstab(cand["clase_predicha"], cand["etiqueta"]))
    sin = cand[cand["etiqueta"] == "fondo"]
    print(f"{len(cand)} candidatos en test, {len(sin)} sin poste etiquetado cerca -> revisar en Street View")
    display(sin.drop(columns="street_view").head(20))

torch.save({"estado": modelo.state_dict(), "ocultas": OCULTAS, "dropout": DROPOUT, "d_in": int(Xtr.shape[1]),
            "n_clases": len(NOMBRES), "clases": list(CLASES), "nombres": NOMBRES,
            "media": MEDIA, "desvio": DESVIO, "umbral": UMBRAL, "pca": str(CACHE_S / "pca.npz"),
            "particion": PARTICION, "mejor_epoca": int(mejor["epoca"]), "ap_val": float(mejor["ap"])},
           SALIDA / f"mlp_{ETIQUETA_MODELO}.pt")

resumen = {"etiqueta_modelo": ETIQUETA_MODELO,
           "modelo": "Sonata (PTv3) congelado, 1088 -> 256 PCA" + (" + 3 rasgos de altura" if RASGOS_ALTURA else "") + ", cabeza MLP " + "x".join(map(str, OCULTAS)) + f" de {len(NOMBRES)} clases",
           "licencia_pesos": "CC-BY-NC 4.0 (uso academico)", "fecha": time.strftime("%Y-%m-%d %H:%M"),
           "particion": PARTICION, "val_de_train": VAL_DE_TRAIN, "clases": NOMBRES[1:], "etiquetas": ETIQUETAS, "postes_compartidos_excluidos": {"val": COMPARTIDOS_VAL, "test": COMPARTIDOS_TEST},
           "config": {"epocas": EPOCAS, "paciencia": PACIENCIA, "lote": LOTE, "lr": LR, "wd": WD, "dropout": DROPOUT,
                      "ocultas": list(OCULTAS), "max_fondo_train": MAX_FONDO_TRAIN, "max_fondo_val": MAX_FONDO_VAL,
                      "pesos_clase": dict(zip(NOMBRES, PESOS_CLASE)),
                      "rondas_negativos_duros": RONDAS_NEGATIVOS_DUROS, "neg_duros_por_bloque": NEG_DUROS_POR_BLOQUE,
                      "franja_neutra_m": FRANJA_NEUTRA,
                      "umbral_sin_etiqueta": UMBRAL_SIN_ETIQUETA, "radio_sin_etiqueta_m": RADIO_SIN_ETIQUETA,
                      "candidatos_sin_etiqueta": len(CANDIDATOS_SIN_ETIQUETA),
                      "clase_objetivo": CLASE_OBJETIVO, "criterio_seleccion": "ap_poste",
                      "fraccion_postes": FRACCION_POSTES, "semilla_fraccion": SEMILLA_FRACCION,
                      "postes_train_usados": POSTES_USADOS,
                      "rasgos_altura": list(pdt.NOMBRES_RASGOS_ALTURA) if RASGOS_ALTURA else [],
                      "semilla": SEMILLA},
           "ronda_final": RONDA_FINAL, "mejor_epoca": int(mejor["epoca"]), "ap_val": float(mejor["ap"]), "umbral": UMBRAL,
           "historia": historia_total.round(5).to_dict("records"),
           "datos": tabla_datos.drop(columns=["GB"]).to_dict("records"),
           "validacion_umbral": tabla_val.round(4).to_dict("records"),
           "test": tabla_test.to_dict("records"), "test_por_clase": tabla_test_clases.to_dict("records"),
           "train_diagnostico": tabla_train.tail(1).to_dict("records"),
           "train_diagnostico_por_clase": tabla_train_clases.to_dict("records")}
(SALIDA / f"resumen_{ETIQUETA_MODELO}.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("->", SALIDA / f"resumen_{ETIQUETA_MODELO}.json")

## 8 · Comparación con los demás cuadernos

Cada cuaderno de modelo deja su `etapa_4_5_modelado_evaluacion/resultados/postes/corrida<N>_<fecha>_<modelo>/resumen_<modelo>.json` con las mismas tablas; aquí se
ponen uno al lado del otro. Solo tiene sentido comparar corridas con la **misma partición y las
mismas etiquetas** (campos `particion` y `etiquetas.md5` de cada JSON).

In [ ]:
display(comparar_resumenes(SALIDA_BASE).round(3))

## Qué mirar al leer los resultados

- **Recuperación de detección en test** es el número del inventario: de cada 100 postes, cuántos
  aparecen. **Precisión** dice cuántos candidatos hay que descartar a mano. La tabla **por clase**
  dice, además, cuántos postes salen con la clase cambiada (`hallados_otra_clase`,
  `clase_equivocada`): para el cliente un poste de MT marcado como BT es un poste hallado pero una
  medición mal asignada.
- **MT se mide con un poste** en test y se aprende con siete. Cualquier cifra de MT es un indicio;
  lo que vale hoy es ver si los siete de `b_1465_43337` se separan de los de BT en el diagnóstico de
  train y si el de `b_1494_43240` sale como MT. Con 20-30 recortes de MT la cifra empezará a
  significar algo.
- Un candidato "sin poste etiquetado cerca" puede ser un falso positivo **o un poste que falta
  por etiquetar**. Los CSV traen el enlace a Street View para decidirlo; en la corrida anterior
  varios candidatos altos de test resultaron ser postes reales sin recortar.
- Si el AP de validación es alto y la recuperación por objeto baja, el cuello de botella es el
  agrupamiento o el filtro de forma (`Z_ESCALA`, `MIN_ALTO_POSTE`... en `postes_datos.py`), no
  la cabeza. La fila "sin filtro" de la tabla de validación es la cota: lo que el modelo halla
  antes de que el filtro tire nada.
- La pérdida de validación que sube mientras el AP se mantiene es sobreconfianza, no pérdida de
  ranking: el modelo memoriza el fondo de train. Regularizar más la cabeza ya se probó (corrida 2)
  y halló menos postes; antes de tocar hiperparámetros conviene validación cruzada por bloques,
  porque 14 postes en validación no alcanzan para elegir modelo ni umbral.